In [23]:
!pip install transformers datasets
!pip install transformers[torch] accelerate -U

In [24]:
!pip install PyGithub datasets

In [25]:
import torch

In [26]:
from github import Github
import re
from datasets import Dataset

In [31]:
g = Github("GITHUB_API_KEY")

/tmp/ipython-input-3937643532.py:1: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github("GITHUB_API_KEY")


##Getting the specific repository from the github

In [30]:
repo = g.get_repo("openai/gym")

It scans the given code as text, finds every line that looks like a Python function definition (def function_name(...) :), and returns a list of the function names.

In [32]:
def extraction_fun_name(code):
    pattern = re.compile(r"def\s+(\w+)\s*\(.*\):")
    functions = pattern.findall(code)
    return functions

It walks through every directory in the repository, finds files ending with .py, and stores them in a list.

In [33]:
python_files =[]
contents = repo.get_contents("")
while contents:
  file_content= contents.pop(0)
  if file_content.type == "dir":
    contents.extend(repo.get_contents(file_content.path))
  elif file_content.path.endswith(".py"):
    python_files.append(file_content)


For every Python file in the repository, it extracts all function names and pairs each function name with the full source code of the file.



In [34]:
data = {"code":[], "function_name":[]}

for file in python_files:
  code=file.decoded_content.decode("utf-8")
  functions=extraction_fun_name(code)
  for function in functions:
    data["code"].append(code)
    data["function_name"].append(function)

In [35]:
dataset=Dataset.from_dict(data)

In [36]:
dataset.save_to_disk("code_generation_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/974 [00:00<?, ? examples/s]

In [37]:
print("Dataset created and saved to disk.")

Dataset created and saved to disk.


It sets up a tokenizer and model, loads a saved dataset, splits it into training and testing sets, enables memory-efficient training, and defines how raw code will be converted into model-ready tokens.

In [38]:
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codegen-350M-mono")
model =AutoModelForCausalLM.from_pretrained("Salesforce/codegen-350M-mono")
tokenizer.pad_token=tokenizer.eos_token
dataset=dataset.load_from_disk("code_generation_dataset")
dataset=dataset.train_test_split(test_size=0.1)
model.gradient_checkpointing_enable()
model.config.use_cache = False

def preprocess_function(examples):
  return tokenizer(
      examples["code"],
      truncation=True,
      max_length=256,
      padding="max_length"
    )


Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

It converts raw code into token IDs, frees unused GPU memory, and defines how the model will be trained while minimizing GPU memory usage.

In [39]:
tokenized_dataset= dataset.map(preprocess_function,batched=True)
torch.cuda.empty_cache()
training_args =TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    save_steps=10_000,
    save_total_limit=2,
    # --- Memory Saving Settings ---
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    dataloader_pin_memory=True
)

Map:   0%|          | 0/876 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

It prepares batches of tokenized code so the model can learn to predict the next token in a sequence, which is how code-generation models are trained.

In [40]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


It connects the model, training configuration, prepared datasets, and data collator, then begins training the model on the code data.

In [41]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
)
trainer.train()

Step,Training Loss


TrainOutput(global_step=219, training_loss=0.3269121701314569, metrics={'train_runtime': 305.1842, 'train_samples_per_second': 2.87, 'train_steps_per_second': 0.718, 'total_flos': 409424602595328.0, 'train_loss': 0.3269121701314569, 'epoch': 1.0})

You give the model the beginning of a function, and it automatically completes the code based on what it learned during training.

In [42]:
def generate_code(prompt, max_length=300):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=max_length
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# test the model with a code generation prompt
prompt = "def binary_search(arr):"
generated_code = generate_code(prompt)

print("Generated Code:")
print(generated_code)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Code:
def binary_search(arr):
    """
    binary_search([0, 1, 2, 3, 4, 5]) -> 4
    binary_search([0, 1, 2, 3, 4, 5]) -> False
    binary_search([0, 1, 2, 3, 4, 5]) -> True
    """
    low = 0
    high = len(arr) - 1
    mid = (low + high) // 2
    while low <= high:
        if arr[mid] == 0:
            low = mid + 1
        elif arr[mid] == 1:
            high = mid - 1
        else:
            return False
        mid = (low + high) // 2
    return True


def test_binary_search():
    assert binary_search([0, 1, 2, 3, 4, 5]) == True
    assert binary_search([0, 1, 2, 3, 4, 5]) == False
    assert binary_search([0, 1, 2, 3, 4, 5]) == True
    assert binary_search([0, 1, 2, 3, 4, 5]) == False


def test_binary_search_recursive():
    assert binary_search_recursive([0, 1, 2, 3, 4, 5]) == True
    assert binary_search_
